# CivOne Decision Log Analysis

Loads `decisions.jsonl`, examines what the AI is actually doing, and fits simple decision trees so you can see which features drive each choice. Once you have enough human-play data mixed in, the same pipeline will train a replacement policy.

In [ ]:
import json
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

LOG_PATH = Path.home() / 'Library/Application Support/CivOne/data/decisions.jsonl'
print(f'Log path: {LOG_PATH}')
print(f'Exists:   {LOG_PATH.exists()}')
if LOG_PATH.exists():
    print(f'Size:     {LOG_PATH.stat().st_size:,} bytes')

In [ ]:
# ── load ─────────────────────────────────────────────────────────────────────
records = []
if LOG_PATH.exists():
    with open(LOG_PATH, encoding='utf-8') as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f'  skipped line {lineno}: {e}')

if not records:
    print('No records yet — play a few turns then re-run.')
else:
    df = pd.DataFrame(records)
    print(f'Records loaded : {len(df):,}')
    print(f'Record types   : {df["type"].value_counts().to_dict()}')
    print(f'Games          : {df["game_id"].nunique()}')
    print(f'Turns covered  : {df["turn"].min()} – {df["turn"].max()}')

## Game outcomes

In [ ]:
outcomes = df[df['type'] == 'game_outcome'].copy()
outcomes[['score', 'turns', 'victory', 'human_won']] = (
    outcomes[['score', 'turns', 'victory', 'human_won']]
    .apply(pd.to_numeric, errors='ignore')
)

if outcomes.empty:
    print('No completed games yet.')
else:
    print(outcomes[['game_id','score','turns','victory','human_won']]
          .sort_values('score', ascending=False)
          .to_string(index=False))
    print(f'\nMean score  : {outcomes["score"].mean():.0f}')
    print(f'Max score   : {outcomes["score"].max()}')
    print(f'Human wins  : {outcomes["human_won"].sum()} / {len(outcomes)}')

# Build a per-game score map for weighting downstream
score_map = dict(zip(outcomes['game_id'], outcomes['score'])) if not outcomes.empty else {}

## Settler decisions

In [ ]:
S_FEATURES = ['food_r2','shield_r2','trade_r2','coastal','river_adj',
              'nearest_city','nearest_own','own_cities']

settlers = df[df['type'] == 'settler'].copy()
for col in S_FEATURES:
    settlers[col] = pd.to_numeric(settlers[col], errors='coerce')
settlers['coastal']   = settlers['coastal'].astype(float)
settlers['river_adj'] = settlers['river_adj'].astype(float)
settlers['game_score'] = settlers['game_id'].map(score_map)
settlers = settlers.dropna(subset=S_FEATURES + ['action'])

print(f'Settler records : {len(settlers):,}')
print(f'\nAction distribution:')
print(settlers['action'].value_counts().to_string())

In [ ]:
# Action mix by terrain type — reveals where the AI tends to found vs. improve
if len(settlers) > 0 and 'terrain' in settlers.columns:
    ct = pd.crosstab(settlers['terrain'], settlers['action'])
    ct['_total'] = ct.sum(axis=1)
    ct = ct.sort_values('_total', ascending=False).drop(columns='_total')
    print('Actions by terrain type (row = terrain, col = action):')
    print(ct.to_string())

In [ ]:
# ── mean feature values per action ───────────────────────────────────────────
if len(settlers) > 1:
    summary = settlers.groupby('action')[S_FEATURES].mean().round(1)
    print('Mean feature values per action:')
    print(summary.to_string())

In [ ]:
# ── decision tree ─────────────────────────────────────────────────────────────
# Needs at least two classes and 30+ samples to be meaningful.
n_classes = settlers['action'].nunique()
n_samples = len(settlers)

if n_classes < 2 or n_samples < 30:
    print(f'Not enough data yet ({n_samples} samples, {n_classes} classes). Keep playing!')
else:
    X = settlers[S_FEATURES].values
    y = settlers['action'].values

    # Weight samples by game score so decisions from better games count more.
    # Fall back to uniform weights if score data is missing.
    if settlers['game_score'].notna().any():
        raw_w = settlers['game_score'].fillna(settlers['game_score'].median()).values
        sample_weight = np.clip(raw_w / raw_w.max(), 0.1, 1.0)
    else:
        sample_weight = None

    X_tr, X_te, y_tr, y_te, w_tr, w_te = train_test_split(
        X, y,
        sample_weight if sample_weight is not None else np.ones(len(X)),
        test_size=0.2, random_state=42
    )

    clf = DecisionTreeClassifier(max_depth=4, min_samples_leaf=max(3, n_samples//40),
                                 class_weight='balanced')
    clf.fit(X_tr, y_tr, sample_weight=w_tr)

    acc = clf.score(X_te, y_te)
    print(f'Held-out accuracy: {acc:.1%}  (baseline: {1/n_classes:.1%} random, {settlers["action"].value_counts(normalize=True).max():.1%} majority)')
    print()
    print(classification_report(y_te, clf.predict(X_te), zero_division=0))

    # Feature importance bar chart
    imp = pd.Series(clf.feature_importances_, index=S_FEATURES).sort_values()
    fig, ax = plt.subplots(figsize=(7, 3.5))
    imp.plot.barh(ax=ax, color='steelblue')
    ax.set_title('Settler decisions — feature importance')
    ax.set_xlabel('Gini importance')
    plt.tight_layout()
    plt.show()

    print('\nTree rules (depth ≤ 4):')
    print(export_text(clf, feature_names=S_FEATURES))

## City production decisions

In [ ]:
P_FEATURES = ['city_size','food_surplus','shields','defenders',
              'nearest_enemy','at_war','own_gold','own_cities']

prod = df[df['type'] == 'city_prod'].copy()
for col in P_FEATURES:
    prod[col] = pd.to_numeric(prod[col], errors='coerce')
prod['at_war']     = prod['at_war'].astype(float)
prod['game_score'] = prod['game_id'].map(score_map)
prod = prod.dropna(subset=P_FEATURES + ['action'])

print(f'City production records: {len(prod):,}')
print(f'\nTop 20 productions chosen:')
print(prod['action'].value_counts().head(20).to_string())

In [ ]:
# Production mix by AI stance
if 'stance' in prod.columns and prod['stance'].notna().any():
    # Collapse rare productions to 'Other' to keep the table readable
    top_n = prod['action'].value_counts().head(10).index
    prod['action_label'] = prod['action'].where(prod['action'].isin(top_n), 'Other')
    ct2 = pd.crosstab(prod['stance'], prod['action_label'])
    print('Production choices by AI stance:')
    print(ct2.to_string())
else:
    print('No stance data yet.')

In [ ]:
n_prod_classes = prod['action'].nunique()
n_prod_samples = len(prod)

if n_prod_classes < 2 or n_prod_samples < 30:
    print(f'Not enough data yet ({n_prod_samples} samples, {n_prod_classes} classes).')
else:
    # Collapse productions with fewer than 5 examples
    counts = prod['action'].value_counts()
    prod['action_g'] = prod['action'].where(counts[prod['action'].values].values >= 5, 'Other')

    Xp = prod[P_FEATURES].values
    yp = prod['action_g'].values

    if prod['game_score'].notna().any():
        raw_wp = prod['game_score'].fillna(prod['game_score'].median()).values
        wp = np.clip(raw_wp / raw_wp.max(), 0.1, 1.0)
    else:
        wp = np.ones(len(Xp))

    Xp_tr, Xp_te, yp_tr, yp_te, wp_tr, _ = train_test_split(
        Xp, yp, wp, test_size=0.2, random_state=42
    )

    clf_p = DecisionTreeClassifier(max_depth=4,
                                   min_samples_leaf=max(3, n_prod_samples//40),
                                   class_weight='balanced')
    clf_p.fit(Xp_tr, yp_tr, sample_weight=wp_tr)

    acc_p = clf_p.score(Xp_te, yp_te)
    print(f'Held-out accuracy: {acc_p:.1%}')
    print()
    print(classification_report(yp_te, clf_p.predict(Xp_te), zero_division=0))

    imp_p = pd.Series(clf_p.feature_importances_, index=P_FEATURES).sort_values()
    fig, ax = plt.subplots(figsize=(7, 3.5))
    imp_p.plot.barh(ax=ax, color='darkorange')
    ax.set_title('City production — feature importance')
    ax.set_xlabel('Gini importance')
    plt.tight_layout()
    plt.show()

    print('\nTree rules (depth ≤ 4):')
    print(export_text(clf_p, feature_names=P_FEATURES))

## Founding quality: what does the terrain look like when the AI founds vs. when it doesn't?

This is the most actionable diagnostic. If founding sites have similar feature distributions to non-founding decisions it means the AI is making poor tradeoffs — settling for mediocre tiles rather than holding out for better ones, or vice versa.

In [ ]:
if len(settlers) >= 10:
    found  = settlers[settlers['action'] == 'found']
    moved  = settlers[settlers['action'] == 'move']
    improved = settlers[settlers['action'].isin(['road','irrigate','mine'])]

    compare_cols = ['food_r2','shield_r2','trade_r2','coastal','river_adj','nearest_city']
    groups = {'Found city': found, 'Moved on': moved, 'Improved tile': improved}
    rows = {name: g[compare_cols].mean() for name, g in groups.items() if len(g) > 0}

    if rows:
        summary_df = pd.DataFrame(rows).T.round(2)
        print('Average feature values by decision type:')
        print(summary_df.to_string())

        fig, axes = plt.subplots(1, len(compare_cols), figsize=(14, 3))
        colors = ['#2196F3', '#FF9800', '#4CAF50']
        for ax, col in zip(axes, compare_cols):
            for i, (name, g) in enumerate(groups.items()):
                if len(g) == 0:
                    continue
                ax.bar(name[:5], g[col].mean(), color=colors[i], alpha=0.8)
            ax.set_title(col, fontsize=8)
            ax.tick_params(axis='x', labelsize=7)
        fig.suptitle('Feature means: found vs. move vs. improve', y=1.02)
        plt.tight_layout()
        plt.show()
    else:
        print('Need a mix of founding and non-founding decisions to compare.')
else:
    print('Not enough settler records yet.')

## Score correlation: which features predict a high-scoring game?

Requires multiple completed games. Each decision is tagged with its game's final score; we measure which decision *features* correlate most strongly with score. A strong positive correlation on `food_r2` for founding decisions, for example, would mean the AI should prefer richer sites.

In [ ]:
if score_map and len(settlers[settlers['game_score'].notna()]) > 20:
    # Founding decisions only — these have the clearest long-term impact
    found_scored = settlers[(settlers['action'] == 'found') & settlers['game_score'].notna()]
    if len(found_scored) > 5:
        corr = found_scored[S_FEATURES + ['game_score']].corr()['game_score'].drop('game_score').sort_values()
        fig, ax = plt.subplots(figsize=(6, 3))
        corr.plot.barh(ax=ax, color=['#d32f2f' if v < 0 else '#388e3c' for v in corr])
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_title('Correlation with game score — founding decisions')
        ax.set_xlabel('Pearson r')
        plt.tight_layout()
        plt.show()
        print(corr.to_string())
    else:
        print('Need more founding decisions across multiple games.')
else:
    print('Need completed games with score data. Keep playing!')

## Export trained models for embedding

Once accuracy is satisfying, save the trees as JSON rule lists. These can be re-implemented in C# without any ML library — a decision tree is just nested if/else. The cell below prints the C#-ready pseudo-code for the settler tree.

In [ ]:
def tree_to_csharp(tree, feature_names, class_names, indent=0):
    """Recursively emit C#-flavoured pseudo-code for a fitted DecisionTreeClassifier."""
    t = tree.tree_
    lines = []

    def recurse(node, depth):
        pad = '    ' * depth
        if t.children_left[node] == t.children_left[0] == -1:  # can't use TREE_LEAF easily
            pass
        if t.children_left[node] != -1:
            feat  = feature_names[t.feature[node]]
            thresh = f'{t.threshold[node]:.3f}'
            lines.append(f'{pad}if ({feat} <= {thresh})')
            lines.append(f'{pad}{{')
            recurse(t.children_left[node],  depth + 1)
            lines.append(f'{pad}}}')
            lines.append(f'{pad}else')
            lines.append(f'{pad}{{')
            recurse(t.children_right[node], depth + 1)
            lines.append(f'{pad}}}')
        else:
            cls_idx = t.value[node][0].argmax()
            cls_name = class_names[cls_idx] if cls_idx < len(class_names) else str(cls_idx)
            n = int(t.value[node][0].sum())
            lines.append(f'{pad}return "{cls_name}";  // n={n}')

    recurse(0, indent)
    return '\n'.join(lines)

try:
    cs_code = tree_to_csharp(clf, S_FEATURES, clf.classes_)
    print('// Settler action — generated decision tree')
    print('// Drop this into AI.Strategy.cs to replace ChooseSettlerImprovement')
    print(cs_code)
except NameError:
    print('Train the settler tree first (run the cells above).')